# DỰ ĐOÁN VÀ ĐÁNH GIÁ NGUY CƠ BỆNH TIM MẠCH (HEALTHCARE AND PATIENT RISK ANALYSIS)

### Nghiên cứu Đa Tập Dữ Liệu (Multi-Dataset Study), Đánh Giá Xuyên Miền (Cross-Domain), Trôi Dữ Liệu (Data Drift) và Khung Dự Đoán Tin Cậy (Trustworthy AI)

---

## 1. Đặt Vấn Đề Khoa Học & Mục Tiêu Nghiên Cứu

Bệnh tim mạch (Cardiovascular Diseases - CVD) là nguyên nhân gây tử vong hàng đầu toàn cầu theo Tổ chức Y tế Thế giới (WHO). Việc ứng dụng Học Máy nhằm phát hiện sớm nguy cơ đóng vai trò cốt lõi trong y học dự phòng. Tuy nhiên, các hệ thống AI y tế thường gặp phải rào cản nghiêm trọng khi chuyển giao từ môi trường nghiên cứu sang thực tế lâm sàng:

1. **Hiện tượng Trôi Dạt Phân Phối (Dataset Shift)**:
   - **Covariate Shift ($P_S(X) \ne P_T(X)$)**: Phân phối nhân khẩu học và sinh trắc (tuổi, huyết áp, tỷ lệ hút thuốc) thay đổi giữa các quốc gia và bệnh viện.
   - **Prior / Base-Rate Shift ($P_S(Y) \ne P_T(Y)$)**: Tỷ lệ hiện mắc bệnh nền khác biệt lớn giữa cộng đồng tầm soát rộng và quần thể bệnh nhân khám chuyên khoa.
   - **Concept Shift ($P_S(Y \mid X) \ne P_T(Y \mid X)$)**: Sự thay đổi bản chất mục tiêu lâm sàng giữa tiên lượng nguy cơ dài hạn (biến cố 10 năm) và chẩn đoán bệnh lý thực thể hiện tại.
2. **Nguy Cơ Tiềm Ẩn Của Các Chỉ Số Đánh Giá Truyền Thống**:
   - Việc sử dụng ngưỡng mặc định $p = 0.50$ và chỉ số Accuracy gây sai lệch chết người trong bối cảnh dữ liệu mất cân bằng lớp và chi phí sai số y tế bất đối xứng (Âm tính giả - False Negative gây nguy cơ tử vong cao hơn nhiều so với Dương tính giả - False Positive).

---

### 1.2. Thu Thập & Nguồn Gốc Dữ Liệu (Data Acquisition & Multi-Cohort Provenance)

Tuân thủ yêu cầu đề tài về việc **tự tìm kiếm và nghiên cứu tối thiểu 2 bộ dữ liệu y tế độc lập**, dự án thu thập và đối chiếu hai tập dữ liệu lâm sàng công khai chuẩn mực quốc tế:

| Tiêu chí so sánh              | Tập A: Framingham Heart Study                                                        | Tập B: Cardiovascular Disease Dataset                                                          |
| :---------------------------- | :----------------------------------------------------------------------------------- | :--------------------------------------------------------------------------------------------- |
| **Nguồn dữ liệu & Bản quyền** | Viện Tim, Phổi & Máu Quốc gia Hoa Kỳ (NHLBI) & ĐH Boston (Kaggle Public)             | Hồ sơ khám lâm sàng Đông Âu (Kaggle / Sulman Sarwar)                                           |
| **Thiết kế nghiên cứu**       | Nghiên cứu thuần tập tiến cứu dài hạn (Longitudinal Cohort Study)                    | Nghiên cứu cắt ngang tại thời điểm khám (Cross-Sectional Examination)                          |
| **Quy mô ban đầu**            | 4.240 bệnh nhân, 15 thuộc tính y tế ban đầu                                          | 70.000 bệnh nhân, 12 thuộc tính sinh trắc & lối sống                                           |
| **Mục tiêu nhãn ($Y$)**       | `TenYearCHD`: Biến cố mạch vành trong 10 năm tới ($1 = \text{Có}, 0 = \text{Không}$) | `cardio`: Hiện diện bệnh lý tim mạch thực thể tại lúc khám ($1 = \text{Có}, 0 = \text{Không}$) |
| **Tỷ lệ nhãn dương ban đầu**  | 15.19% (644 / 4.240 ca dương tính) – **Mất cân bằng lớp thực tế**                    | 49.47% (34.979 / 70.000 ca dương tính) – **Cân bằng tự nhiên**                                 |
| **Đặc trưng thu thập**        | Nhân trắc học, tiền sử bệnh án, thói quen sinh hoạt, huyết áp, hóa sinh máu          | Nhân trắc học, thói quen sinh hoạt, huyết áp, phân tầng cholesterol & glucose                  |
| **Định dạng & Lưu trữ**       | `data/raw/framingham.csv` (CSV phân tách dấu phẩy)                                   | `data/raw/cardio_train.csv` (CSV phân tách dấu chấm phẩy `;`)                                  |

#### Cơ sở khoa học & Chiến lược thu thập dữ liệu đa nguồn:

1. **Kiểm định khả năng thích ứng với Trôi dạt phân phối (Dataset Shift)**: Thay vì chỉ đánh giá trên 1 tập dữ liệu duy nhất (dễ gây ngộ nhận về độ chính xác do Overfitting vào một quần thể), việc thu thập từ 2 vùng dịch tễ độc lập (Mỹ vs Đông Âu) cho phép kiểm chứng độ bền vững thực sự của mô hình học máy.
2. **Kịch bản kiểm thử chuyển giao khắt khe (Cross-Domain Transfer)**: Tập A mang tính chất _tiên lượng rủi ro tương lai_, Tập B mang tính chất _chẩn đoán thực thể hiện tại_. Sự đối lập này giúp đánh giá năng lực của các biểu diễn đặc trưng lâm sàng cốt lõi khi chuyển giao không hiệu chỉnh (Zero-shot Transfer).
3. **Đảm bảo tính hội tụ về Lược đồ lâm sàng chung**: Dù xuất phát từ hai nghiên cứu độc lập, cả hai tập đều thu thập 8 chỉ số tim mạch nền tảng theo khuyến cáo WHO/AHA: Tuổi, Giới tính, Huyết áp tâm thu/tâm trương, BMI, Cholesterol, Glucose, và Hút thuốc lá.

---

### 1.3. Thiết Kế Thực Nghiệm Chuẩn Mực Trong Nghiên Cứu Này:

- **Hai Quần Thể Y Tế Độc Lập**:
  - **Tập A: Framingham Heart Study (Mỹ)**: $N=4.240$ bệnh nhân, tỷ lệ bệnh $15.19\%$, bài toán tiên lượng biến cố mạch vành 10 năm (`TenYearCHD`).
  - **Tập B: Cardiovascular Disease Dataset (Đông Âu)**: $N=70.000$ bệnh nhân ($68.619$ sau làm sạch sinh học), tỷ lệ bệnh $49.47\%$, bài toán chẩn đoán hiện diện bệnh tim mạch (`cardio`).
- **So Sánh Toàn Diện 7 Thuật Toán Học Máy Kinh Điển**:
  - Baseline (`most_frequent`), Logistic Regression ($L_2$), $K$-Nearest Neighbors (KNN), Gaussian Naive Bayes, Decision Tree / Random Forest, Support Vector Machines (SVM RBF & Kernel Approximation Nyström), và XGBoost.
- **Quy Chuẩn Đánh Giá Chống Rò Rỉ & Kiểm Định Độ Tin Cậy**:
  - Phân chia $60/20/20$ Stratified theo từng bệnh nhân; đóng gói toàn bộ khâu chuẩn hóa trong `Pipeline` độc lập.
  - Tối ưu hóa ngưỡng quyết định lâm sàng $p^*$ theo bài toán tối thiểu hóa chi phí $\text{Cost} = 1 \times FP + 5 \times FN$ với ràng buộc $\text{Recall} \ge 0.80$.
  - Ước lượng khoảng tin cậy $95\%$ bằng Percentile Bootstrap ($B=2000$).
- **Phương Pháp Luận Tiên Tiến Mở Rộng**:
  - Định lượng trôi dạt dữ liệu bằng **Adversarial Validation**.
  - Khắc phục sai lệch xác suất xuyên miền bằng **Bidirectional Post-hoc Probability Calibration** (Platt Scaling & Isotonic Regression).
  - Giải thích cơ chế quyết định và bóc tách ca Âm tính giả bằng **TreeExplainer SHAP**.
  - Bảo chứng thống kê không phụ thuộc phân phối bằng **Conformal Prediction via MAPIE** ($1-\alpha = 90\%$).


In [ ]:
# ==============================================================================
# BƯỚC 1: KHỞI TẠO MÔI TRƯỜNG & THIẾT LẬP TÍNH TÁI LẬP (REPRODUCIBILITY)
# - Cấu hình hạt giống ngẫu nhiên (SEED=42)
# - Import thư viện Scikit-learn, XGBoost, MAPIE, SHAP
# - Khởi tạo thư mục và hàm lưu trữ biểu đồ nghiên cứu
# ==============================================================================

import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Sklearn imports
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
    learning_curve,
    train_test_split,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Optional scikit-learn 1.6+ FrozenEstimator support
try:
    from sklearn.frozen import FrozenEstimator
except ImportError:
    FrozenEstimator = None

# XGBoost, SHAP & MAPIE
import mapie
import shap
import xgboost as xgb
from xgboost import XGBClassifier

try:
    from mapie.classification import SplitConformalClassifier
except ImportError:
    from mapie.classification import MapieClassifier as SplitConformalClassifier

# Global configuration
warnings.filterwarnings("ignore")
np.random.seed(42)
plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)
plt.rcParams.update(
    {
        "font.sans-serif": "DejaVu Sans",
        "axes.edgecolor": "#cccccc",
        "axes.linewidth": 0.8,
    }
)

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)


def save_fig(fig, filename, dpi=200):
    path = os.path.join(FIG_DIR, filename)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f" Saved figure: {path} (DPI={dpi})")


print(
    f"Environment ready | Python {sys.version.split()[0]} | "
    f"sklearn {sklearn.__version__} | xgboost {xgb.__version__} | "
    f"shap {shap.__version__} | mapie {mapie.__version__}"
)

## 2. Tiền Xử Lý Dữ Liệu & Chuẩn Hóa Lược Đồ (Data Preprocessing & Schema Harmonization)

Hai tập dữ liệu xuất phát từ các nguồn dịch tễ học độc lập với quy cách đo lường khác biệt. Quá trình tiền xử lý được thiết kế nghiêm ngặt theo các tiêu chuẩn y khoa và nguyên lý biểu diễn không gian đặc trưng:

### 2.1. Lọc Ngoại Lai Theo Ngưỡng Sinh Tồn Y Khoa (Biological Plausibility Filtering)

Trên Tập B (`Cardio`), xuất hiện các bản ghi nhập liệu sai lệch nghiêm trọng về mặt sinh học. Áp dụng quy tắc lọc:

- **Huyết áp tâm thu (`ap_hi`)**: Giới hạn trong khoảng $[60, 250]$ mmHg. Ngưỡng dưới $60$ mmHg tương ứng với tình trạng sốc tuần hoàn mất bù; ngưỡng trên $250$ mmHg là cơn tăng huyết áp cấp cứu cực độ.
- **Huyết áp tâm trương (`ap_lo`)**: Giới hạn trong khoảng $[30, 160]$ mmHg.
- **Điều kiện sinh lý bắt buộc**: $\text{ap\_hi} > \text{ap\_lo}$ (Huyết áp tâm thu luôn phải lớn hơn huyết áp tâm trương do chênh lệch áp lực mạch máu).
- **Chỉ số nhân trắc học**: Chiều cao trong $[120, 220]$ cm; Cân nặng trong $[30, 200]$ kg.

### 2.2. Đồng Bộ Lược Đồ Về 8 Đặc Trưng Lâm Sàng Chung Theo 3 Nhóm Biểu Diễn

Để đảm bảo tính tương thích toán học cho các mô hình dựa trên khoảng cách (KNN, SVM RBF) và mô hình tuyến tính có điều chuẩn (Logistic Regression), các biến được phân tách thành 3 nhóm xử lý riêng biệt trong `ColumnTransformer`:

1. **Nhóm Đặc Trưng Liên Tục (`NUM_FEATURES`)** — Điền khuyết bằng `median`, chuẩn hóa z-score qua `StandardScaler`:
   - `age`: Tuổi sinh học (quy đổi chính xác về năm thực: chia $365.25$ đối với dữ liệu theo ngày của Cardio).
   - `bmi`: Chỉ số khối cơ thể tính theo công thức Quetelet: $\text{BMI} = \frac{\text{weight (kg)}}{(\text{height (m)})^2}$.
   - `sys_bp`: Huyết áp tâm thu (mmHg).
   - `dia_bp`: Huyết áp tâm trương (mmHg).
2. **Nhóm Đặc Trưng Thứ Bậc (`ORD_FEATURES`)** — Điền khuyết bằng `most_frequent`, chuẩn hóa qua `StandardScaler`:
   - `chol_level`: Phân tầng theo khuyến cáo của NCEP ATP III ($1$: Bình thường $<200$ mg/dL, $2$: Ranh giới cao $200\text{–}239$ mg/dL, $3$: Nguy cơ cao $\ge 240$ mg/dL).
   - `gluc_level`: Phân tầng theo tiêu chuẩn ADA ($1$: Bình thường $<100$ mg/dL, $2$: Rối loạn dung nạp/Tiền đái tháo đường $100\text{–}125$ mg/dL, $3$: Đái tháo đường $\ge 126$ mg/dL).
     _(Cơ sở chuẩn hóa)_: Việc đưa các thang bậc $1, 2, 3$ về cùng thang đo đơn vị phương sai giúp hàm khoảng cách Euclidean không bị chi phối áp đảo bởi thang đo số học của các biến khác.
3. **Nhóm Đặc Trưng Nhị Phân (`BIN_FEATURES`)** — Điền khuyết bằng `most_frequent`, giữ nguyên mã hóa $\{0, 1\}$:
   - `is_male`: Giới tính sinh học ($1 = \text{Nam}, 0 = \text{Nữ}$).
   - `smoker`: Tình trạng hút thuốc lá hiện tại ($1 = \text{Có}, 0 = \text{Không}$).

### 2.3. Chiến Lược Ngăn Chặn Rò Rỉ Dữ Liệu (Leakage Prevention Protocol)

Dữ liệu được phân chia phân tầng $60\% / 20\% / 20\%$ (Train / Val / Test). Mọi phép tính thống kê (trung vị, giá trị trung bình, độ lệch chuẩn) chỉ được học (`fit`) duy nhất trên tập Train, sau đó áp dụng phép biến đổi (`transform`) sang tập Validation và Test qua `Pipeline`. Tuyệt đối không thực hiện bất kỳ phép tiền xử lý nào trên toàn bộ tập dữ liệu trước khi phân chia.


In [ ]:
# ==============================================================================
# BƯỚC 2.1: TẢI DỮ LIỆU & LỌC NGOẠI LAI SINH HỌC BẤT KHẢ THI (DATA HYGIENE)
# - Tải 2 tập: Framingham (Mỹ) & Cardio Train (Đông Âu)
# - Loại bỏ các giá trị huyết áp âm hoặc ngoài ngưỡng sinh tồn y khoa
# ==============================================================================

df_framingham_raw = pd.read_csv("data/raw/framingham.csv")
df_cardio_raw = pd.read_csv("data/raw/cardio_train.csv", sep=";")

print(f"Raw Framingham shape: {df_framingham_raw.shape}")
print(f"Raw Cardio shape:     {df_cardio_raw.shape}")

# Biological sanity filtering for Cardio dataset
c = df_cardio_raw
valid_cardio = (
    c["ap_hi"].between(60, 250)
    & c["ap_lo"].between(30, 160)
    & (c["ap_hi"] > c["ap_lo"])
    & c["height"].between(120, 220)
    & c["weight"].between(30, 200)
)
df_cardio_filtered = df_cardio_raw[valid_cardio].copy()
removed_rows = len(df_cardio_raw) - len(df_cardio_filtered)
print(
    f"Cardio filtered: Removed {removed_rows} invalid physiological rows ({removed_rows / len(df_cardio_raw) * 100:.2f}%). Remaining: {len(df_cardio_filtered)}"
)

In [ ]:
# ==============================================================================
# BƯỚC 2.2: CHUẨN HÓA VÀ ĐỒNG BỘ LƯỢC ĐỒ ĐẶC TRƯNG (SCHEMA HARMONIZATION)
# - Quy đổi độ tuổi sang năm thực
# - Chuẩn hóa huyết áp và BMI theo công thức lâm sàng
# - Phân loại đặc trưng: Liên tục (NUM), Thứ bậc (ORD), Nhị phân (BIN)
# ==============================================================================

NUM_FEATURES = ["age", "bmi", "sys_bp", "dia_bp"]
ORD_FEATURES = ["chol_level", "gluc_level"]
BIN_FEATURES = ["is_male", "smoker"]
COMMON_FEATURES = NUM_FEATURES + ORD_FEATURES + BIN_FEATURES


def harmonize_framingham(df):
    clean = pd.DataFrame(
        {
            "age": df["age"].astype(float),
            "bmi": df["BMI"].astype(float),
            "sys_bp": df["sysBP"].astype(float),
            "dia_bp": df["diaBP"].astype(float),
            "is_male": df["male"].astype(int),
            "smoker": df["currentSmoker"].astype(int),
            "chol_level": np.select(
                [df["totChol"].isna(), df["totChol"] < 200, df["totChol"] < 240],
                [np.nan, 1.0, 2.0],
                default=3.0,
            ),
            "gluc_level": np.select(
                [df["glucose"].isna(), df["glucose"] < 100, df["glucose"] < 126],
                [np.nan, 1.0, 2.0],
                default=3.0,
            ),
            "target": df["TenYearCHD"].astype(int),
        },
        index=df.index,
    )
    return clean[COMMON_FEATURES + ["target"]]


def harmonize_cardio(df):
    clean = pd.DataFrame(
        {
            "age": (df["age"] / 365.25).round(1),
            "bmi": (df["weight"] / ((df["height"] / 100.0) ** 2)).round(2),
            "sys_bp": df["ap_hi"].astype(float),
            "dia_bp": df["ap_lo"].astype(float),
            "is_male": (df["gender"] == 2).astype(int),
            "smoker": df["smoke"].astype(int),
            "chol_level": df["cholesterol"].astype(float),
            "gluc_level": df["gluc"].astype(float),
            "target": df["cardio"].astype(int),
        },
        index=df.index,
    )
    return clean[COMMON_FEATURES + ["target"]]


df_A = harmonize_framingham(df_framingham_raw)
df_B = harmonize_cardio(df_cardio_filtered)

print("--- HARMONIZED DATASET OVERVIEW ---")
print(
    f"Dataset A (Framingham): {df_A.shape[0]} rows | Target Rate: {df_A['target'].mean() * 100:.2f}% (Positives: {df_A['target'].sum()})"
)
print(
    f"Dataset B (Cardio):     {df_B.shape[0]} rows | Target Rate: {df_B['target'].mean() * 100:.2f}% (Positives: {df_B['target'].sum()})"
)
print(f"Numeric Features ({len(NUM_FEATURES)}): {NUM_FEATURES}")
print(f"Ordinal Features ({len(ORD_FEATURES)}): {ORD_FEATURES}")
print(f"Binary Features  ({len(BIN_FEATURES)}): {BIN_FEATURES}")

In [ ]:
# ==============================================================================
# BƯỚC 2.3: KHÁM PHÁ DỮ LIỆU (EDA) & KIỂM ĐỊNH TỶ LỆ DỮ LIỆU KHUYẾT THIẾU
# - Thống kê phần trăm khuyết thiếu trên các đặc trưng chung
# - Khảo sát phân phối lâm sàng và ma trận tương quan giữa 2 tập dữ liệu
# ==============================================================================

missing_audit = pd.DataFrame(
    {
        "Missing_A_Count": df_A[COMMON_FEATURES].isna().sum(),
        "Missing_A_Pct(%)": (df_A[COMMON_FEATURES].isna().mean() * 100).round(2),
        "Missing_B_Count": df_B[COMMON_FEATURES].isna().sum(),
        "Missing_B_Pct(%)": (df_B[COMMON_FEATURES].isna().mean() * 100).round(2),
    }
)
print("--- MISSING VALUE AUDIT ---")
display(missing_audit)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, col in zip(axes.flatten(), COMMON_FEATURES):
    if col in NUM_FEATURES:
        for df, lbl, colr in [
            (df_A, "Dataset A (Framingham)", "#1f77b4"),
            (df_B, "Dataset B (Cardio)", "#ff7f0e"),
        ]:
            sns.kdeplot(
                data=df[col].dropna(),
                ax=ax,
                label=lbl,
                color=colr,
                fill=True,
                alpha=0.3,
                lw=2,
            )
        ax.set_title(f"Distribution: {col}", fontsize=13, fontweight="bold")
    else:
        cnt_A = df_A[col].value_counts(normalize=True).sort_index()
        cnt_B = df_B[col].value_counts(normalize=True).sort_index()
        cats = sorted(list(set(cnt_A.index) | set(cnt_B.index)))
        x, bar_w = np.arange(len(cats)), 0.35
        ax.bar(
            x - bar_w / 2,
            [cnt_A.get(c, 0) for c in cats],
            bar_w,
            label="Dataset A",
            color="#1f77b4",
            alpha=0.85,
        )
        ax.bar(
            x + bar_w / 2,
            [cnt_B.get(c, 0) for c in cats],
            bar_w,
            label="Dataset B",
            color="#ff7f0e",
            alpha=0.85,
        )
        ax.set_xticks(x)
        ax.set_xticklabels([str(int(c)) if not pd.isna(c) else "NA" for c in cats])
        ax.set_title(f"Proportion: {col}", fontsize=13, fontweight="bold")
    ax.legend(loc="upper right", frameon=True)

plt.tight_layout()
save_fig(fig, "fig1_eda_distributions.png")
plt.show()

# Correlation Heatmap for Both Datasets
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
for ax, df, title, cmap in [
    (ax1, df_A, "Correlation Heatmap: Dataset A (Framingham)", "Blues"),
    (ax2, df_B, "Correlation Heatmap: Dataset B (Cardio)", "Oranges"),
]:
    sns.heatmap(
        df[COMMON_FEATURES + ["target"]].corr(),
        annot=True,
        fmt=".2f",
        cmap=cmap,
        ax=ax,
        cbar=True,
        square=True,
    )
    ax.set_title(title, fontsize=14, fontweight="bold")

plt.tight_layout()
save_fig(fig, "fig2_correlation_matrix.png")
plt.show()

In [ ]:
# ==============================================================================
# BƯỚC 2.4: PHÂN CHIA DỮ LIỆU PHÂN TẦNG (STRATIFIED TRAIN / VAL / TEST: 60/20/20)
# - Đơn vị độc lập: 1 bệnh nhân / 1 hàng (Ngăn chặn hoàn toàn rò rỉ dữ liệu)
# - Xây dựng Pipeline tiền xử lý: Imputation, Scaling, One-Hot Encoding
# ==============================================================================


def split_data(df, target_col="target", random_state=42):
    X, y = df[COMMON_FEATURES].copy(), df[target_col].copy()
    X_train, X_tmp, y_train, y_tmp = train_test_split(
        X, y, test_size=0.40, stratify=y, random_state=random_state
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


X_train_A, X_val_A, X_test_A, y_train_A, y_val_A, y_test_A = split_data(df_A)
X_train_B, X_val_B, X_test_B, y_train_B, y_val_B, y_test_B = split_data(df_B)

for name, (X_tr, y_tr, X_v, y_v, X_te, y_te) in [
    ("Dataset A", (X_train_A, y_train_A, X_val_A, y_val_A, X_test_A, y_test_A)),
    ("Dataset B", (X_train_B, y_train_B, X_val_B, y_val_B, X_test_B, y_test_B)),
]:
    print(
        f"{name} Splits -> Train: {len(X_tr)} ({y_tr.mean() * 100:.1f}% pos) | Val: {len(X_v)} ({y_v.mean() * 100:.1f}% pos) | Test: {len(X_te)} ({y_te.mean() * 100:.1f}% pos)"
    )


def make_preprocessor():
    # 3-branch ColumnTransformer (num, ord, bin) per pipeline
    imp_scale = lambda strat: Pipeline(
        [("imputer", SimpleImputer(strategy=strat)), ("scaler", StandardScaler())]
    )
    return ColumnTransformer(
        transformers=[
            ("num", imp_scale("median"), NUM_FEATURES),
            ("ord", imp_scale("most_frequent"), ORD_FEATURES),
            (
                "bin",
                Pipeline([("imputer", SimpleImputer(strategy="most_frequent"))]),
                BIN_FEATURES,
            ),
        ],
        verbose_feature_names_out=False,
    )


print(
    "Preprocessing factory make_preprocessor() initialized (3-branch metric-safe & state-isolated)."
)

## 3. Huấn Luyện & Đánh Giá Mô Hình Nội Miền (In-Domain Model Assessment)

Quá trình huấn luyện khảo sát toàn diện 7 họ thuật toán đại diện cho các trường phái học máy khác nhau. Mọi siêu tham số tối ưu đều được lựa chọn thông qua `GridSearchCV` kết hợp 5-Fold Stratified Cross-Validation trên tập Train:

1. **Dummy Baseline**: Bộ phân loại cơ sở theo chiến lược `most_frequent` (luôn dự đoán nhãn chiếm đa số) để thiết lập cận dưới hiệu năng.
2. **Logistic Regression ($L_2$-regularized)**: Mô hình xác suất tuyến tính. Tinh chỉnh hệ số nghịch đảo điều chuẩn $C \in [0.01, 10.0]$. Trích xuất tỷ số số phận $\text{Odds Ratio} = \exp(w_j)$ phục vụ phân tích lâm sàng.
3. **$K$-Nearest Neighbors (KNN)**: Phương pháp phi tham số dựa trên khoảng cách. Tinh chỉnh số láng giềng $k \in [5, 31]$, trọng số khoảng cách (`uniform`, `distance`) và chuẩn khoảng cách Minkowski ($p=1$ Manhattan, $p=2$ Euclidean).
4. **Gaussian Naive Bayes**: Mô hình xác suất sinh dựa trên định lý Bayes với giả thiết các đặc trưng độc lập có điều kiện khi biết nhãn lớp. Đánh giá tính vững thông qua độ lệch chuẩn CV 5-fold.
5. **Decision Tree & Random Forest**: Tập hợp bagging các cây quyết định độc lập nhằm giảm phương sai. Tinh chỉnh độ sâu tối đa `max_depth` $\in [4, 12]$ và số mẫu tối thiểu ở nút lá `min_samples_leaf` $\in [5, 20]$.
6. **XGBoost Classifier**: Mô hình Gradient Boosting hiện đại. Tinh chỉnh `max_depth` và `learning_rate`. Sử dụng kỹ thuật **Early Stopping** (`early_stopping_rounds=30`) trên lát cắt kiểm thực nội bộ $10\%$ tách từ tập Train (hoàn toàn cách ly với tập Validation chính) để xác định điểm dừng huấn luyện tối ưu của `n_estimators`, ngăn chặn Overfitting triệt để.
7. **Support Vector Machines (SVM)**:
   - _Trên Tập A ($N=4.240$)_: Huấn luyện Kernel SVM RBF đầy đủ (`SVC(kernel="rbf", probability=True)`), sử dụng `class_weight="balanced"` nhằm tối ưu hóa siêu phẳng phân tách trong điều kiện lớp thiểu số $15.19\%$. Xác suất hậu nghiệm được ước lượng thông qua Platt Scaling nội tại 5-fold CV.
   - _Trên Tập B ($N=70.000$)_: Do chi phí tính toán ma trận Gram của Kernel SVM chuẩn tăng theo cấp số $O(N^2)$ đến $O(N^3)$, ta áp dụng kỹ thuật xấp xỉ hạt nhân **Nyström Kernel Approximation** (ánh xạ dữ liệu sang không gian đặc trưng hữu hạn với hàm nhân RBF) kết hợp với **Stochastic Gradient Descent (`SGDClassifier`)**. Sử dụng hàm mất mát `loss="log_loss"` để thu được xác suất hậu nghiệm thực sự, cho phép huấn luyện với độ phức tạp tuyến tính $O(N)$ mà vẫn bảo toàn năng lực phi tuyến.

### 3.1. Phương Pháp Luận Đánh Giá Lâm Sàng (Clinical Evaluation Protocol)

Trong sàng lọc y tế, sai lầm loại II (Âm tính giả - False Negative) gây hậu quả nghiêm trọng hơn nhiều so với sai lầm loại I (Dương tính giả - False Positive). Do đó, quy trình đánh giá tuân thủ 3 nguyên tắc:

1. **Hàm Chi Phí Y Tế Bất Đối Xứng (Asymmetric Medical Cost Function)**:
   $$\text{Cost} = 1.0 \times FP + 5.0 \times FN$$
   (Mỗi ca bỏ sót bệnh bị phạt nặng gấp 5 lần một ca chỉ định xét nghiệm thừa).
2. **Tối Ưu Hóa Ngưỡng Cắt Xác Suất Lâm Sàng ($p^*$)**:
   Thay vì áp dụng ngưỡng cơ học $0.50$, ta tìm ngưỡng $p^* \in [0.05, 0.95]$ trên tập Validation sao cho:
   $$\text{Tìm } p^* = \arg\max_{p} \{ \text{Precision}(p) \mid \text{Recall}(p) \ge 0.80 \}$$
   Sau khi xác định $p^*$ tối ưu trên Validation, ngưỡng này được giữ nguyên cố định để kiểm định độc lập trên tập Test.
3. **Đánh Giá Khoảng Tin Cậy 95% Bằng Percentile Bootstrap**:
   Thực hiện lấy mẫu hoàn lại $B=2000$ lần trên tập Test để ước lượng phân phối của ROC-AUC. Loại trừ và báo cáo cụ thể các mẫu suy biến (nếu có).
4. **Khảo Sát Động Thái Học Tập (Learning Curves)**:
   Đánh giá sự hội tụ của ROC-AUC theo kích thước tập huấn luyện từ $20\%$ đến $100\%$ trên 5-Fold Stratified CV kèm dải phương sai $\pm 1\sigma$.


In [ ]:
# ==============================================================================
# BƯỚC 3.1: KHỞI TẠO TẬP HỢP MÔ HÌNH (MODEL ZOO) & KHÔNG GIAN SIÊU THAM SỐ
# - Khởi tạo 7 thuật toán: Dummy, Logistic, KNN, Naive Bayes, Decision Tree, RF, XGB
# - Thiết lập Search Space cho quá trình Hyperparameter Tuning
# ==============================================================================


def get_model_zoo(dataset_name="A"):
    models = {
        "Dummy_Baseline": DummyClassifier(strategy="most_frequent"),
        "Logistic_Regression": LogisticRegression(
            penalty="l2", max_iter=1000, random_state=42
        ),
        "KNN": KNeighborsClassifier(),
        "Gaussian_NB": GaussianNB(),
        "Random_Forest": RandomForestClassifier(
            n_estimators=100, random_state=42, n_jobs=-1
        ),
        "XGBoost": XGBClassifier(
            n_estimators=100, eval_metric="logloss", random_state=42, n_jobs=-1
        ),
    }
    param_grids = {
        "Dummy_Baseline": {},
        "Logistic_Regression": {"classifier__C": [0.01, 0.1, 1.0, 10.0]},
        "KNN": (
            {
                "classifier__n_neighbors": [5, 11, 15, 25],
                "classifier__weights": ["uniform", "distance"],
                "classifier__p": [1, 2],
            }
            if dataset_name == "A"
            else {
                "classifier__n_neighbors": [11, 15, 31],
                "classifier__weights": ["uniform", "distance"],
                "classifier__p": [2],
            }
        ),
        "Gaussian_NB": {},
        "Random_Forest": {
            "classifier__max_depth": [4, 6, 8, 12],
            "classifier__min_samples_leaf": [1, 5],
        },
        "XGBoost": {
            "classifier__max_depth": [2, 3, 4],
            "classifier__learning_rate": [0.03, 0.05, 0.1],
        },
    }

    if dataset_name == "A":
        models["SVM"] = SVC(
            kernel="rbf", probability=True, class_weight="balanced", random_state=42
        )
        param_grids["SVM"] = {
            "classifier__C": [0.1, 1.0, 10.0],
            "classifier__gamma": ["scale", 0.01, 0.1],
        }
    else:
        linear_clf = SGDClassifier(loss="log_loss", penalty="l2", random_state=42)
        models["SVM"] = Pipeline(
            [
                ("nystroem", Nystroem(kernel="rbf", n_components=100, random_state=42)),
                ("clf", linear_clf),
            ]
        )
        param_grids["SVM"] = {
            "classifier__nystroem__gamma": ["scale", 0.1],
            "classifier__clf__alpha": [1e-4, 1e-3, 1e-2],
        }

    return models, param_grids


models_A, grids_A = get_model_zoo("A")
models_B, grids_B = get_model_zoo("B")

print(
    f"Initialized {len(models_A)} models + CV grids for Dataset A and {len(models_B)} models + CV grids for Dataset B."
)

In [ ]:
# ==============================================================================
# BƯỚC 3.2: ENGINE ĐÁNH GIÁ, CHỌN MÔ HÌNH VÀ TỐI ƯU NGƯỠNG Y TẾ (OPTIMAL THRESHOLD)
# - Định nghĩa hàm chi phí y tế (FN phạt nặng gấp 5 lần FP)
# - Tối ưu ngưỡng cắt xác suất p* để đảm bảo Độ nhạy (Recall >= 80%)
# - Ước lượng khoảng tin cậy 95% Bootstrap CI cho ROC-AUC
# ==============================================================================


def compute_medical_cost(y_true, y_pred, cost_fp=1.0, cost_fn=5.0):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return fp * cost_fp + fn * cost_fn


def find_best_threshold(y_val, y_val_proba, min_recall=0.80):
    precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_proba)
    precisions, recalls = precisions[:-1], recalls[:-1]

    valid_mask = recalls >= min_recall
    if np.any(valid_mask):
        valid_prec = precisions[valid_mask]
        max_prec = np.max(valid_prec)
        best_threshold = np.max(thresholds[valid_mask][valid_prec == max_prec])
    else:
        P = float(np.sum(y_val))
        tp = recalls * P
        fn = P - tp
        fp = np.where(precisions > 0, tp * (1.0 / precisions - 1.0), 0.0)
        best_threshold = thresholds[np.argmin(fp + 5.0 * fn)]
    return float(np.clip(best_threshold, 0.05, 0.95))


def bootstrap_roc_auc_ci(
    y_true, y_proba, n_bootstraps=2000, alpha=0.05, random_state=42
):
    rng = np.random.RandomState(random_state)
    bootstrapped_scores, failed_reps = [], 0
    y_true_arr, y_proba_arr = np.array(y_true), np.array(y_proba)
    n = len(y_true_arr)

    for _ in range(n_bootstraps):
        idx = rng.randint(0, n, n)
        if len(np.unique(y_true_arr[idx])) < 2:
            failed_reps += 1
            continue
        bootstrapped_scores.append(roc_auc_score(y_true_arr[idx], y_proba_arr[idx]))

    sorted_scores = np.sort(bootstrapped_scores)
    return (
        float(np.percentile(sorted_scores, (alpha / 2.0) * 100)),
        float(np.percentile(sorted_scores, (1.0 - alpha / 2.0) * 100)),
        failed_reps,
    )


def select_xgb_n_estimators_early_stopping(
    preproc_factory, best_clf_params, X_train, y_train
):
    X_tr_es, X_val_es, y_tr_es, y_val_es = train_test_split(
        X_train, y_train, test_size=0.10, stratify=y_train, random_state=42
    )
    prep = preproc_factory()
    xgb_es = XGBClassifier(
        n_estimators=1000,
        max_depth=best_clf_params.get("classifier__max_depth", 4),
        learning_rate=best_clf_params.get("classifier__learning_rate", 0.05),
        eval_metric="logloss",
        early_stopping_rounds=30,
        random_state=42,
        n_jobs=-1,
    )
    xgb_es.fit(
        prep.fit_transform(X_tr_es),
        y_tr_es,
        eval_set=[(prep.transform(X_val_es), y_val_es)],
        verbose=False,
    )
    return int(xgb_es.best_iteration + 1)


def evaluate_pipeline(
    models_dict,
    param_grids_dict,
    preproc_factory,
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
    dataset_name="Dataset",
):
    trained_pipelines, test_probas, optimal_thresholds = {}, {}, {}
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    results = []

    print(
        f"\n{'=' * 30} CV MODEL SELECTION & TEST EVALUATION ON {dataset_name} {'=' * 30}"
    )
    for name, clf in models_dict.items():
        base_pipe = Pipeline(
            [("preprocessor", preproc_factory()), ("classifier", clone(clf))]
        )
        grid = param_grids_dict.get(name, {})

        if grid:
            search = GridSearchCV(
                base_pipe, grid, cv=cv, scoring="roc_auc", n_jobs=-1, refit=True
            )
            search.fit(X_train, y_train)
            best_idx = search.best_index_
            cv_mean = float(search.cv_results_["mean_test_score"][best_idx])
            cv_std = float(search.cv_results_["std_test_score"][best_idx])
            best_params, pipe = search.best_params_, search.best_estimator_

            if name == "XGBoost":
                best_n_trees = select_xgb_n_estimators_early_stopping(
                    preproc_factory, best_params, X_train, y_train
                )
                pipe = Pipeline(
                    [
                        ("preprocessor", preproc_factory()),
                        (
                            "classifier",
                            clone(clf).set_params(
                                n_estimators=best_n_trees,
                                max_depth=best_params["classifier__max_depth"],
                                learning_rate=best_params["classifier__learning_rate"],
                            ),
                        ),
                    ]
                )
                es_scores = cross_val_score(
                    pipe, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1
                )
                cv_mean, cv_std = float(np.mean(es_scores)), float(np.std(es_scores))
                pipe.fit(X_train, y_train)
                best_params = {
                    **best_params,
                    "classifier__n_estimators(ES)": best_n_trees,
                }
        else:
            cv_scores = cross_val_score(
                base_pipe, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1
            )
            cv_mean, cv_std = float(np.mean(cv_scores)), float(np.std(cv_scores))
            best_params, pipe = {"default": True}, base_pipe.fit(X_train, y_train)

        trained_pipelines[name] = pipe
        val_proba = pipe.predict_proba(X_val)[:, 1]
        test_proba = pipe.predict_proba(X_test)[:, 1]
        test_probas[name] = test_proba

        best_thresh = (
            0.5
            if name == "Dummy_Baseline"
            else find_best_threshold(y_val, val_proba, min_recall=0.80)
        )
        optimal_thresholds[name] = best_thresh

        test_pred_def = (test_proba >= 0.50).astype(int)
        test_pred_opt = (test_proba >= best_thresh).astype(int)

        if len(np.unique(test_proba)) > 1:
            auc = roc_auc_score(y_test, test_proba)
            ci_low, ci_high, failed_reps = bootstrap_roc_auc_ci(
                y_test, test_proba, n_bootstraps=2000
            )
        else:
            auc, ci_low, ci_high, failed_reps = 0.5, 0.5, 0.5, 0

        ap = average_precision_score(y_test, test_proba)
        rec_opt = recall_score(y_test, test_pred_opt)
        prec_opt = precision_score(y_test, test_pred_opt, zero_division=0)
        f1_opt = f1_score(y_test, test_pred_opt, zero_division=0)
        cost_opt = compute_medical_cost(y_test, test_pred_opt)
        cost_def = compute_medical_cost(y_test, test_pred_def)
        brier = brier_score_loss(y_test, test_proba)

        clean_params = ", ".join(
            f"{k.replace('classifier__', '')}={v}" for k, v in best_params.items()
        )
        results.append(
            {
                "Model": name,
                "Best_Params": clean_params,
                "CV_ROC_AUC": f"{cv_mean:.3f} +/- {cv_std:.3f}",
                "ROC_AUC": auc,
                "AUC_95_CI": f"[{ci_low:.3f}, {ci_high:.3f}]",
                "Boot_Fail": failed_reps,
                "PR_AUC": ap,
                "Threshold_Opt": best_thresh,
                "Recall_Opt": rec_opt,
                "Precision_Opt": prec_opt,
                "F1_Opt": f1_opt,
                "Cost_Default(0.5)": cost_def,
                "Cost_Tuned(p*)": cost_opt,
                "Cost_Reduction": f"{((cost_def - cost_opt) / max(cost_def, 1)) * 100:.1f}%",
                "Brier_Score": brier,
            }
        )
        print(
            f"Finished {name:20s} | CV: {cv_mean:.3f}+/-{cv_std:.3f} | Test AUC: {auc:.3f} (fail={failed_reps}) | "
            f"PR-AUC: {ap:.3f} | p*: {best_thresh:.3f} | Recall@p*: {rec_opt:.3f} | Cost: {cost_def:.0f} -> {cost_opt:.0f}"
        )

    return pd.DataFrame(results), trained_pipelines, test_probas, optimal_thresholds

In [ ]:
# ==============================================================================
# BƯỚC 3.3: HUẤN LUYỆN VÀ ĐÁNH GIÁ TOÀN DIỆN MÔ HÌNH NỘI MIỀN (IN-DOMAIN EXECUTION)
# - Thực thi Cross-Validation & GridSearch trên Dataset A (Framingham)
# - Thực thi Cross-Validation & GridSearch trên Dataset B (Cardio)
# ==============================================================================

df_results_A, pipes_A, probas_A, thresh_A = evaluate_pipeline(
    models_A,
    grids_A,
    make_preprocessor,
    X_train_A,
    y_train_A,
    X_val_A,
    y_val_A,
    X_test_A,
    y_test_A,
    "Dataset A (Framingham)",
)

df_results_B, pipes_B, probas_B, thresh_B = evaluate_pipeline(
    models_B,
    grids_B,
    make_preprocessor,
    X_train_B,
    y_train_B,
    X_val_B,
    y_val_B,
    X_test_B,
    y_test_B,
    "Dataset B (Cardio)",
)

In [ ]:
# ==============================================================================
# BƯỚC 3.4: TRỰC QUAN HÓA ĐƯỜNG CONG ROC VÀ PR (PRECISION-RECALL CURVES)
# - Vẽ và lưu biểu đồ ROC Curve & PR Curve cho cả hai tập dữ liệu
# - Đánh giá năng lực phân loại của 7 mô hình trong điều kiện mất cân bằng lớp
# ==============================================================================


def plot_curves(y_test, probas_dict, dataset_label, filename):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    colors = [
        "#7f7f7f",
        "#1f77b4",
        "#2ca02c",
        "#d62728",
        "#9467bd",
        "#8c564b",
        "#e377c2",
    ]

    for (name, proba), color in zip(probas_dict.items(), colors):
        if name == "Dummy_Baseline":
            continue
        fpr, tpr, _ = roc_curve(y_test, proba)
        auc = roc_auc_score(y_test, proba)
        ax1.plot(fpr, tpr, lw=2, color=color, label=f"{name} (AUC = {auc:.3f})")

        prec, rec, _ = precision_recall_curve(y_test, proba)
        ap = average_precision_score(y_test, proba)
        ax2.plot(rec, prec, lw=2, color=color, label=f"{name} (PR-AUC = {ap:.3f})")

    ax1.plot([0, 1], [0, 1], "k--", lw=1.5, alpha=0.7)
    ax1.set(
        title=f"ROC Curves - {dataset_label}",
        xlabel="False Positive Rate (FPR)",
        ylabel="True Positive Rate (TPR / Recall)",
    )
    ax1.legend(loc="lower right", frameon=True)

    ax2.axhline(
        y=y_test.mean(),
        color="k",
        linestyle="--",
        lw=1.5,
        alpha=0.7,
        label=f"No-skill Baseline ({y_test.mean():.2f})",
    )
    ax2.set(
        title=f"Precision-Recall Curves - {dataset_label}",
        xlabel="Recall",
        ylabel="Precision",
    )
    ax2.legend(loc="upper right", frameon=True)

    plt.tight_layout()
    save_fig(fig, filename)
    plt.show()


plot_curves(
    y_test_A, probas_A, "Dataset A (Framingham)", "fig3_roc_pr_curves_framingham.png"
)
plot_curves(y_test_B, probas_B, "Dataset B (Cardio)", "fig4_roc_pr_curves_cardio.png")

In [ ]:
# ==============================================================================
# BƯỚC 3.5: PHÂN TÍCH ĐƯỜNG CONG HỌC TẬP (LEARNING CURVES)
# - Đánh giá Bias-Variance Tradeoff với 5-Fold Stratified CV
# - Xác định tình trạng Overfitting / Underfitting khi tăng kích thước tập huấn luyện
# ==============================================================================

cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_sizes_grid = np.linspace(0.2, 1.0, 5)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
models_lc = [
    (
        "Logistic_Regression",
        "Learning Curves: Logistic Regression (Dataset A)",
        [0.65, 0.85],
    ),
    ("XGBoost", "Learning Curves: XGBoost (Dataset A)", [0.60, 1.00]),
]

for ax, (m_name, title, ylim) in zip(axes, models_lc):
    sizes, tr_scores, val_scores = learning_curve(
        pipes_A[m_name],
        X_train_A,
        y_train_A,
        cv=cv_strat,
        scoring="roc_auc",
        train_sizes=train_sizes_grid,
        n_jobs=-1,
    )
    tr_m, tr_s = np.mean(tr_scores, axis=1), np.std(tr_scores, axis=1)
    val_m, val_s = np.mean(val_scores, axis=1), np.std(val_scores, axis=1)

    ax.plot(sizes, tr_m, "o-", color="#1f77b4", lw=2, label="Training ROC-AUC")
    ax.fill_between(sizes, tr_m - tr_s, tr_m + tr_s, alpha=0.15, color="#1f77b4")
    ax.plot(
        sizes, val_m, "s--", color="#ff7f0e", lw=2, label="5-Fold CV ROC-AUC (+/- 1 SD)"
    )
    ax.fill_between(sizes, val_m - val_s, val_m + val_s, alpha=0.15, color="#ff7f0e")

    ax.set(
        title=title, xlabel="Training Sample Size", ylabel="ROC-AUC Score", ylim=ylim
    )
    ax.legend(loc="lower right", frameon=True)

plt.tight_layout()
save_fig(fig, "fig5_learning_curves.png")
plt.show()

In [ ]:
# ==============================================================================
# BƯỚC 3.6: BẢNG TỔNG HỢP HIỆU NĂNG VÀ BIỂU ĐỒ SO SÁNH NỘI MIỀN (IN-DOMAIN SUMMARY)
# - Hiển thị bảng số liệu ROC-AUC (kèm 95% CI), PR-AUC, Recall@p*, Chi phí y tế
# - Vẽ biểu đồ cột so sánh tổng thể các mô hình trên 2 tập dữ liệu
# ==============================================================================

print("--- SUMMARY TABLE: DATASET A (FRAMINGHAM) ---")
display(df_results_A.sort_values(by="ROC_AUC", ascending=False))

print("\n--- SUMMARY TABLE: DATASET B (CARDIO) ---")
display(df_results_B.sort_values(by="ROC_AUC", ascending=False))

# Comparison Bar Chart
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
models_plot = df_results_A[df_results_A["Model"] != "Dummy_Baseline"]["Model"].values
x, bar_w = np.arange(len(models_plot)), 0.35

chart_configs = [
    (
        axes[0],
        "ROC_AUC",
        "ROC-AUC Score",
        [0.5, 0.88],
        "In-Domain ROC-AUC Comparison Across Models",
    ),
    (
        axes[1],
        "F1_Opt",
        "F1 Score at Optimal Medical Threshold (p*)",
        [0.0, 0.88],
        "In-Domain F1 Score at Medical Threshold (p*)",
    ),
]

for ax, metric, ylabel, ylim, title in chart_configs:
    v_A = [
        df_results_A[df_results_A["Model"] == m][metric].values[0] for m in models_plot
    ]
    v_B = [
        df_results_B[df_results_B["Model"] == m][metric].values[0] for m in models_plot
    ]
    ax.bar(
        x - bar_w / 2,
        v_A,
        bar_w,
        label="Dataset A (Framingham)",
        color="#1f77b4",
        alpha=0.85,
    )
    ax.bar(
        x + bar_w / 2,
        v_B,
        bar_w,
        label="Dataset B (Cardio)",
        color="#ff7f0e",
        alpha=0.85,
    )
    ax.set_xticks(x)
    ax.set_xticklabels(models_plot, rotation=25, ha="right", fontsize=11)
    ax.set(ylabel=ylabel, ylim=ylim, title=title)
    ax.legend(loc="upper left", frameon=True)

plt.tight_layout()
save_fig(fig, "fig6_in_domain_comparison.png")
plt.show()

In [ ]:
# ==============================================================================
# BƯỚC 3.7: PHÂN TÍCH TỶ SỐ SỐ PHẬN (ODDS RATIOS) CỦA MÔ HÌNH LOGISTIC REGRESSION
# - Ước lượng exp(w_j) thể hiện mối tương quan có điều kiện với nguy cơ bệnh
# - Lưu ý khoa học: Tương quan thống kê không đồng nhất với quan hệ nhân quả (Causation)
# ==============================================================================


def extract_odds_ratios(pipe_lr):
    prep, clf = pipe_lr.named_steps["preprocessor"], pipe_lr.named_steps["classifier"]
    scales = {
        f: float(s)
        for b in ["num", "ord"]
        for f, s in zip(
            prep.transformers_[0 if b == "num" else 1][2],
            prep.named_transformers_[b].named_steps["scaler"].scale_,
        )
    }
    return pd.DataFrame(
        [
            {
                "Feature": f,
                "Coef": round(float(w), 4),
                "OR_1SD": round(float(np.exp(w)), 4),
                "OR_Unit": round(float(np.exp(w / scales.get(f, 1.0))), 4),
            }
            for f, w in zip(prep.get_feature_names_out(), clf.coef_[0])
        ]
    )


df_odds_cmp = extract_odds_ratios(pipes_A["Logistic_Regression"]).merge(
    extract_odds_ratios(pipes_B["Logistic_Regression"]),
    on="Feature",
    suffixes=("_A", "_B"),
)
print(
    "\n--- LOGISTIC REGRESSION ODDS RATIOS (exp(w_j)) | NOTE: Association != Causation ---"
)
display(df_odds_cmp)

## 4. Thực Nghiệm Xuyên Miền (Cross-Domain), Phân Tích Lỗi & Khung Dự Đoán Tin Cậy

Trong thực tế triển khai, mô hình học máy hiếm khi hoạt động trong môi trường đóng. Sự khác biệt về phân phối dữ liệu giữa các cơ sở y tế thường dẫn đến sự suy giảm nghiêm trọng về độ chính xác lâm sàng. Phần này thực hiện 5 phân tích chuyên sâu nhằm phát hiện, định lượng và khắc phục các rủi ro này:

### 4.1. Đánh Giá Chuyển Giao Xuyên Miền Không Hiệu Chỉnh (Zero-Shot Cross-Domain Transfer)

Kiểm tra tính suy giảm hiệu năng khi áp dụng chéo mô hình huấn luyện từ Tập A (Framingham, Mỹ) sang kiểm thử trên Tập B (Cardio, Đông Âu) và ngược lại ($A \to B$ và $B \to A$).
Sự sụt giảm hiệu năng được phân rã thành ba thành phần toán học:

1. **Covariate Shift**: $P_A(X) \ne P_B(X)$ — Do đặc điểm dịch tễ và sinh trắc học của hai quần thể khác nhau.
2. **Prior Probability / Base-Rate Shift**: $P_A(Y=1) \approx 0.1519$ trong khi $P_B(Y=1) \approx 0.4947$. Sự chênh lệch tỷ lệ hiện mắc khiến ngưỡng tối ưu nguồn hoàn toàn sai lệch khi áp sang tập đích.
3. **Target Concept Shift**: $P_A(Y \mid X) \ne P_B(Y \mid X)$ — Tập A mô hình hóa nguy cơ biến cố động mạch vành 10 năm (`TenYearCHD`), trong khi Tập B chẩn đoán tình trạng bệnh tim mạch hiện diện tại thời điểm khám (`cardio`). Mối liên kết giữa các chỉ số sinh lý $X$ và nhãn $Y$ có bản chất cơ chế bệnh sinh khác nhau.

---

### 4.2. Định Lượng Trôi Dữ Liệu Bằng Kiểm Định Đối Nghịch (Adversarial Validation)

- **Phương pháp luận**: Gộp không gian đặc trưng của hai tập dữ liệu và gán nhãn miền giả định: $Y_{\text{domain}} = 0$ đối với Tập A và $Y_{\text{domain}} = 1$ đối với Tập B.
- Huấn luyện bộ phân loại `XGBClassifier` với 5-Fold Stratified CV để dự đoán nguồn gốc dữ liệu.
- **Nguyên lý kiểm định**:
  - Nếu hai quần thể đồng nhất về phân phối: Bộ phân loại không thể phân biệt nguồn gốc $\implies \text{ROC-AUC} \approx 0.50$.
  - Nếu hai quần thể phân ly rõ rệt (Data Drift nghiêm trọng): $\text{ROC-AUC} \to 1.0$.
  - Tầm quan trọng đặc trưng (Gain Feature Importance) của mô hình đối nghịch cho phép định lượng chính xác các chỉ số lâm sàng có mức độ trôi lệch cao nhất.

---

### 4.3. Hiệu Chuẩn Xác Suất Hậu Nghiệm Hai Chiều (Bidirectional Post-hoc Calibration)

Khi chuyển giao mô hình xuyên miền, sai số xác suất (Brier Score) tăng vọt do hiện tượng _Under-confidence_ (chiều $A \to B$) hoặc _Over-confidence_ (chiều $B \to A$). Ta áp dụng kỹ thuật hiệu chuẩn hậu nghiệm trên lát cắt Validation của miền đích:

- **Platt Scaling (Sigmoid)**: Ước lượng hàm hồi quy logistic ánh xạ điểm logit của mô hình nguồn sang xác suất thực tế trên miền đích: $P(Y=1 \mid f) = \frac{1}{1 + \exp(A \cdot f + B)}$.
- **Isotonic Regression**: Phương pháp phi tham số tối ưu hóa hàm đơn điệu từng đoạn bằng thuật toán Pool Adjacent Violators Algorithm (PAVA).
- Đánh giá sự phục hồi của Brier Score và độ nhạy lâm sàng sau khi kết hợp hiệu chuẩn với tái tối ưu ngưỡng trên miền đích.

---

### 4.4. Giải Thích Quyết Định Mô Hình Toàn Cục & Phân Tích Âm Tính Giả Với SHAP

- **Cơ sở lý thuyết**: Áp dụng Shapley Additive exPlanations (`shap.TreeExplainer`) dựa trên lý thuyết trò chơi hợp tác nhằm phân rã dự đoán của mô hình cây thành đóng góp biên của từng đặc trưng:
  $$f(x) = \phi_0 + \sum_{j=1}^M \phi_j(x)$$
- **So sánh Đa Miền Toàn Cục (Global Multi-Domain SHAP)**: So sánh trọng số $\mathbb{E}[|\phi_j|]$ giữa quần thể Mỹ và Đông Âu để làm sáng tỏ sự khác biệt trong cơ chế gây bệnh.
- **Nghiên Cứu Điển Hình Ca Âm Tính Giả Nguy Hiểm (Local False Negative Case Study)**: Bóc tách trường hợp bệnh nhân thực tế mang bệnh nhưng bị mô hình dự đoán xác suất cực thấp, trực quan hóa qua biểu đồ thác nước (Waterfall Plot) để giải mã nguyên nhân mô hình bị đánh lừa.

---

### 4.5. Dự Đoán Phù Hợp Đảm Bảo Độ Phủ Thống Kê (Conformal Prediction via MAPIE)

Trong các ứng dụng y tế trọng yếu, các dự đoán điểm (Point Prediction) đơn lẻ không cung cấp thông tin về độ bất định nhận thức (Epistemic Uncertainty). Kỹ thuật **Split Conformal Prediction** cho phép thiết lập tập dự đoán $C(X) \subseteq \{0, 1\}$ với bảo chứng thống kê hữu hạn không phụ thuộc phân phối (Distribution-free Guarantee):
$$P(Y \in C(X)) \ge 1 - \alpha$$
Với mức sai số chấp nhận $\alpha = 0.10$ (đảm bảo độ bao phủ thực nghiệm $\ge 90\%$):

- $C(X) = \{0\}$: Phân loại bệnh nhân an toàn, loại trừ nguy cơ với độ tin cậy $90\%$.
- $C(X) = \{1\}$: Khẳng định nguy cơ cao, đưa vào diện theo dõi điều trị.
- $C(X) = \{0, 1\}$: **Vùng Bất Định (Uncertain Region)** — Mô hình nhận diện trường hợp phức tạp, tự động chuyển hồ sơ cho bác sĩ chuyên khoa hội chẩn.


In [ ]:
# ==============================================================================
# BƯỚC 4.1: ĐÁNH GIÁ CHUYỂN GIAO XUYÊN MIỀN KHÔNG CẦN CHỈNH SỬA (ZERO-SHOT TRANSFER)
# - Kiểm tra độ suy giảm hiệu năng: Mô hình học từ Mỹ test trên Đông Âu và ngược lại
# - So sánh độ nhạy Recall và Chi phí y tế trước/sau khi chuyển giao
# ==============================================================================


def eval_cross_domain(
    model_name, pipe_src, X_tgt, y_tgt, src_name, tgt_name, thresh_src
):
    tgt_proba = pipe_src.predict_proba(X_tgt)[:, 1]
    pred_def = (tgt_proba >= 0.50).astype(int)
    pred_thresh = (tgt_proba >= thresh_src).astype(int)

    return {
        "Transfer": f"{src_name} -> {tgt_name}",
        "Model": model_name,
        "Cross_ROC_AUC": roc_auc_score(y_tgt, tgt_proba),
        "Cross_PR_AUC": average_precision_score(y_tgt, tgt_proba),
        "Recall@0.5": recall_score(y_tgt, pred_def),
        "Recall@Src_Thresh": recall_score(y_tgt, pred_thresh),
        "Cost@0.5": compute_medical_cost(y_tgt, pred_def),
        "Cost@Src_Thresh": compute_medical_cost(y_tgt, pred_thresh),
        "Brier_Score": brier_score_loss(y_tgt, tgt_proba),
    }


cross_results = []
for m in ["Logistic_Regression", "Random_Forest", "XGBoost"]:
    cross_results.append(
        eval_cross_domain(
            m, pipes_A[m], X_test_B, y_test_B, "Dataset A", "Dataset B", thresh_A[m]
        )
    )
    cross_results.append(
        eval_cross_domain(
            m, pipes_B[m], X_test_A, y_test_A, "Dataset B", "Dataset A", thresh_B[m]
        )
    )

df_cross = pd.DataFrame(cross_results)
print("--- ZERO-SHOT CROSS-DOMAIN EVALUATION RESULTS ---")
display(df_cross)

print(
    "Note (B -> A Cost@Src_Thresh > Cost@0.5): Balanced Cardio thresholds (thresh_B ~ 0.38-0.39) "
    "are too low for imbalanced Framingham (15.2% pos) under over-confident B -> A probabilities, "
    "inflating False Positives and raising total Cost."
)

In [ ]:
# ==============================================================================
# BƯỚC 4.2: KIỂM ĐỊNH ĐỐI NGHỊCH ĐO LƯỜNG TRÔI DỮ LIỆU (ADVERSARIAL VALIDATION)
# - Huấn luyện bộ phân loại phân biệt Domain A vs Domain B (Leakage-free)
# - Phân tích ROC-AUC của Domain Classifier và các đặc trưng gây trôi dạt lớn nhất
# ==============================================================================

X_adv = pd.concat(
    [X_train_A, X_train_B.sample(n=len(X_train_A), random_state=42)],
    ignore_index=True,
)
y_domain = np.repeat([0, 1], len(X_train_A))

# Split BEFORE preprocessing to prevent data leakage into test fold
X_adv_tr, X_adv_te, y_adv_tr, y_adv_te = train_test_split(
    X_adv, y_domain, test_size=0.33, stratify=y_domain, random_state=42
)

adv_pipe = Pipeline(
    [
        ("preprocessor", make_preprocessor()),
        (
            "classifier",
            XGBClassifier(
                n_estimators=100,
                max_depth=3,
                learning_rate=0.05,
                eval_metric="logloss",
                random_state=42,
            ),
        ),
    ]
)
adv_pipe.fit(X_adv_tr, y_adv_tr)

adv_probs = adv_pipe.predict_proba(X_adv_te)[:, 1]
adv_auc = roc_auc_score(y_adv_te, adv_probs)

print(
    f"\n{'*' * 50}\nADVERSARIAL VALIDATION ROC-AUC: {adv_auc:.4f}\n"
    + (
        "CONCLUSION: Severe distribution shift detected between Dataset A (Framingham) and Dataset B (Cardio)!"
        if adv_auc > 0.70
        else "CONCLUSION: Distributions are relatively homogenous."
    )
    + f"\n{'*' * 50}\n"
)

feat_names = adv_pipe.named_steps["preprocessor"].get_feature_names_out().tolist()
importances = adv_pipe.named_steps["classifier"].feature_importances_
indices = np.argsort(importances)[::-1]

print("Top Drift Features (Gain Importance):")
for rank, idx in enumerate(indices, 1):
    print(f"  {rank}. {feat_names[idx]:12s}: {importances[idx]:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(range(len(feat_names)), importances[indices[::-1]], color="#d62728", alpha=0.85)
ax.set_yticks(range(len(feat_names)))
ax.set_yticklabels([feat_names[i] for i in indices[::-1]], fontsize=11)
ax.set(
    xlabel="Adversarial Feature Importance (Gain)",
    title=f"Adversarial Validation: Features Driving Population Drift (AUC = {adv_auc:.3f})",
)

plt.tight_layout()
save_fig(fig, "fig7_adversarial_validation_drift.png")
plt.show()

In [ ]:
# ==============================================================================
# BƯỚC 4.3: HIỆU CHUẨN XÁC SUẤT HẬU KỲ HAI CHIỀU (POST-HOC PROBABILITY CALIBRATION)
# - Áp dụng Isotonic Regression và Platt Scaling (Sigmoid) trên tập Validation đích
# - Đánh giá sự cải thiện của Brier Score và độ tin cậy của xác suất dự báo
# ==============================================================================


def fit_calibrators(pipe_src, X_val_tgt, y_val_tgt):
    est_kw = (
        {"estimator": FrozenEstimator(pipe_src)}
        if FrozenEstimator is not None
        else {"estimator": pipe_src, "cv": "prefit"}
    )
    c_iso = CalibratedClassifierCV(**est_kw, method="isotonic").fit(
        X_val_tgt, y_val_tgt
    )
    c_sig = CalibratedClassifierCV(**est_kw, method="sigmoid").fit(X_val_tgt, y_val_tgt)
    return c_iso, c_sig


def eval_cal_row(setting, y_true, proba, threshold=0.50):
    pred = (proba >= threshold).astype(int)
    return {
        "Setting": setting,
        "Threshold": threshold,
        "Brier_Score": brier_score_loss(y_true, proba),
        "Recall": recall_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "Cost": compute_medical_cost(y_true, pred),
    }


cal_data, cal_clinical_rows = {}, []
print("--- BIDIRECTIONAL BRIER SCORE BEFORE & AFTER CALIBRATION ---")

directions = [
    ("A -> B", pipes_A["XGBoost"], X_val_B, y_val_B, X_test_B, y_test_B),
    ("B -> A", pipes_B["XGBoost"], X_val_A, y_val_A, X_test_A, y_test_A),
]

for dir_lbl, pipe_src, X_v, y_v, X_te, y_te in directions:
    c_iso, c_sig = fit_calibrators(pipe_src, X_v, y_v)
    probs = {
        "Raw XGBoost": pipe_src.predict_proba(X_te)[:, 1],
        "Sigmoid / Platt": c_sig.predict_proba(X_te)[:, 1],
        "Isotonic": c_iso.predict_proba(X_te)[:, 1],
    }
    cal_data[dir_lbl] = probs

    b_raw, b_sig, b_iso = (
        brier_score_loss(y_te, probs["Raw XGBoost"]),
        brier_score_loss(y_te, probs["Sigmoid / Platt"]),
        brier_score_loss(y_te, probs["Isotonic"]),
    )
    print(
        f"Transfer {dir_lbl} | Raw: {b_raw:.4f} | "
        f"Sigmoid: {b_sig:.4f} (-{(b_raw - b_sig) / b_raw * 100:.1f}%) | "
        f"Isotonic: {b_iso:.4f} (-{(b_raw - b_iso) / b_raw * 100:.1f}%)"
    )

    for name, p in [
        ("Raw", probs["Raw XGBoost"]),
        ("Sigmoid", probs["Sigmoid / Platt"]),
        ("Isotonic", probs["Isotonic"]),
    ]:
        cal_clinical_rows.append(
            eval_cal_row(f"{dir_lbl} | {name} @ 0.50", y_te, p, 0.50)
        )

    # Split X_val_tgt 50/50 for leakage-free calibration + target threshold optimization (Recall >= 0.80)
    Xv_fit, Xv_th, yv_fit, yv_th = train_test_split(
        X_v, y_v, test_size=0.50, stratify=y_v, random_state=42
    )
    c_iso_s, c_sig_s = fit_calibrators(pipe_src, Xv_fit, yv_fit)
    for name, c_est in [("Sigmoid", c_sig_s), ("Isotonic", c_iso_s)]:
        p_star = find_best_threshold(
            yv_th, c_est.predict_proba(Xv_th)[:, 1], min_recall=0.80
        )
        cal_clinical_rows.append(
            eval_cal_row(
                f"{dir_lbl} | {name} @ p* (cal)",
                y_te,
                c_est.predict_proba(X_te)[:, 1],
                p_star,
            )
        )

df_cal_clinical = pd.DataFrame(cal_clinical_rows)
print("\n--- CLINICAL SCREENING METRICS BEFORE & AFTER CALIBRATION (XGBoost) ---")
display(df_cal_clinical)

In [ ]:
# ==============================================================================
# BƯỚC 4.4: TRỰC QUAN HÓA ĐƯỜNG CONG HIỆU CHUẨN (CALIBRATION CURVES)
# - Vẽ Reliability Diagrams (Đường cong độ tin cậy) so sánh mô hình gốc vs hiệu chuẩn
# - Minh họa trực quan việc khắc phục hiện tượng Overconfident / Underconfident
# ==============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
curve_configs = [
    (
        axes[0],
        y_test_B,
        cal_data["A -> B"],
        "Cross-Domain A -> B (Framingham -> Cardio)",
        10,
        "uniform",
    ),
    (
        axes[1],
        y_test_A,
        cal_data["B -> A"],
        "Cross-Domain B -> A (Cardio -> Framingham, Quantile Bins)",
        8,
        "quantile",
    ),
]

for ax, y_te, p_dict, title, n_bins, strat in curve_configs:
    for (lbl, p), col in zip(p_dict.items(), ["#d62728", "#1f77b4", "#2ca02c"]):
        prob_true, prob_pred = calibration_curve(y_te, p, n_bins=n_bins, strategy=strat)
        ax.plot(
            prob_pred,
            prob_true,
            marker="o",
            lw=2,
            color=col,
            label=f"{lbl} (Brier = {brier_score_loss(y_te, p):.3f})",
        )
    ax.plot([0, 1], [0, 1], "k--", label="Perfect Calibration", alpha=0.7)
    ax.set(
        title=title,
        xlabel="Mean Predicted Probability",
        ylabel="Empirical True Fraction of Positives",
    )
    ax.legend(loc="upper left", frameon=True)

plt.tight_layout()
save_fig(fig, "fig8_calibration_curves.png")
plt.show()

In [ ]:
# ==============================================================================
# BƯỚC 4.5: GIẢI THÍCH MÔ HÌNH TOÀN CỤC VỚI SHAP (CROSS-POPULATION COMPARISON)
# - Tính toán TreeExplainer SHAP values trên tập kiểm tra của cả 2 tập dữ liệu
# - So sánh mức độ quan trọng (Feature Importance) giữa bệnh nhân Mỹ vs Đông Âu
# ==============================================================================


def extract_shap(pipe, X_data):
    preproc = pipe.named_steps["preprocessor"]
    X_trans = preproc.transform(X_data)
    exp = shap.TreeExplainer(pipe.named_steps["classifier"])(X_trans)
    cols = preproc.get_feature_names_out().tolist()
    return X_trans, exp, cols, dict(zip(cols, np.mean(np.abs(exp.values), axis=0)))


X_test_A_trans, shap_explanation_A, shap_cols_A, mean_shap_A = extract_shap(
    pipes_A["XGBoost"], X_test_A
)
_, shap_explanation_B, _, mean_shap_B = extract_shap(
    pipes_B["XGBoost"], X_test_B.sample(n=2000, random_state=42)
)

df_shap_cmp = pd.DataFrame(
    {
        "Feature": shap_cols_A,
        "Dataset A (Framingham - US)": [mean_shap_A[f] for f in shap_cols_A],
        "Dataset B (Cardio - Eastern Europe)": [mean_shap_B[f] for f in shap_cols_A],
    }
).sort_values(by="Dataset A (Framingham - US)", ascending=True)

print("--- MULTI-DOMAIN MEAN |SHAP| FEATURE IMPORTANCE ---")
display(df_shap_cmp.sort_values(by="Dataset A (Framingham - US)", ascending=False))

# 2-Panel Plot: Left = Cross-population Comparison, Right = Dataset A Beeswarm
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6.5))
y_idx, bar_h = np.arange(len(df_shap_cmp)), 0.38
ax1.barh(
    y_idx - bar_h / 2,
    df_shap_cmp["Dataset A (Framingham - US)"],
    bar_h,
    label="Dataset A: Framingham (US)",
    color="#1f77b4",
    alpha=0.85,
)
ax1.barh(
    y_idx + bar_h / 2,
    df_shap_cmp["Dataset B (Cardio - Eastern Europe)"],
    bar_h,
    label="Dataset B: Cardio (Eastern Europe)",
    color="#ff7f0e",
    alpha=0.85,
)
ax1.set_yticks(y_idx)
ax1.set_yticklabels(df_shap_cmp["Feature"], fontsize=11)
ax1.set(
    xlabel="Mean |SHAP Value| (Average Impact Magnitude)",
    title="Multi-Domain SHAP Comparison: US vs Eastern Europe",
)
ax1.legend(loc="lower right", frameon=True)

plt.sca(ax2)
shap.summary_plot(
    shap_explanation_A.values,
    X_test_A_trans,
    feature_names=shap_cols_A,
    show=False,
    plot_size=None,
)
ax2.set_title(
    "SHAP Value Polarity (Dataset A - Framingham)", fontsize=13, fontweight="bold"
)

plt.tight_layout()
save_fig(fig, "fig9_shap_global_summary.png")
plt.show()

In [ ]:
# ==============================================================================
# BƯỚC 4.6: PHÂN TÍCH CỤC BỘ CA BỆNH BỎ SÓT NGUY HIỂM (FALSE NEGATIVE CASE STUDY)
# - Tìm kiếm bệnh nhân có bệnh nhưng mô hình dự đoán xác suất thấp nhất (Worst FN)
# - Phân tích SHAP Waterfall để bóc tách lý do tại sao mô hình bỏ sót bệnh nhân này
# ==============================================================================

y_pred_A_opt = (probas_A["XGBoost"] >= thresh_A["XGBoost"]).astype(int)
fn_indices = np.where((y_test_A.values == 1) & (y_pred_A_opt == 0))[0]
print(
    f"Total False Negatives on Test A: {len(fn_indices)} patients out of {y_test_A.sum()} actual positive cases."
)

if len(fn_indices) > 0:
    worst_fn_idx = fn_indices[np.argmin(probas_A["XGBoost"][fn_indices])]
    p_raw = X_test_A.iloc[worst_fn_idx]
    print(
        f"Analyzing Worst False Negative Patient (Index #{worst_fn_idx}): Pred={probas_A['XGBoost'][worst_fn_idx]:.3f} vs True=1\n"
        f"Patient Profile: Age={p_raw['age']}, BP={p_raw['sys_bp']}/{p_raw['dia_bp']}, "
        f"BMI={p_raw['bmi']}, Male={int(p_raw['is_male'])}, "
        f"Smoker={int(p_raw['smoker'])}, Chol={int(p_raw['chol_level'])}, Gluc={int(p_raw['gluc_level'])}"
    )

    fig = plt.figure(figsize=(10, 6))
    shap_sample = shap_explanation_A[worst_fn_idx]
    shap_sample.feature_names = shap_cols_A
    shap.plots.waterfall(shap_sample, show=False)
    plt.title(
        f"SHAP Waterfall: Why Model Missed Patient #{worst_fn_idx} (False Negative)",
        fontsize=13,
        fontweight="bold",
        pad=15,
    )
    plt.tight_layout()
    save_fig(plt.gcf(), "fig10_shap_local_false_negatives.png")
    plt.show()

In [ ]:
# ==============================================================================
# BƯỚC 4.7: DỰ ĐOÁN TIN CẬY VỚI PHƯƠNG PHÁP CHUẨN HÌNH (CONFORMAL PREDICTION VIA MAPIE)
# - Thiết lập ngưỡng phủ đảm bảo 90% (Alpha = 0.10) không phụ thuộc phân phối dữ liệu
# - Đánh giá tỷ lệ tập dự đoán rỗng/hai nhãn và độ bao phủ thực nghiệm (Coverage)
# ==============================================================================

ALPHA = 0.10  # 90% guaranteed coverage


def run_conformal_eval(pipe_model, X_cal, y_cal, X_test, y_test, dataset_label):
    try:
        mapie_clf = SplitConformalClassifier(
            estimator=pipe_model, confidence_level=1.0 - ALPHA, prefit=True
        )
        mapie_clf.conformalize(X_cal, y_cal)
        _, y_ps = mapie_clf.predict_set(X_test)
    except Exception:
        mapie_clf = SplitConformalClassifier(
            estimator=pipe_model, method="lac", cv="prefit"
        )
        mapie_clf.fit(X_cal, y_cal)
        _, y_ps = mapie_clf.predict(X_test, alpha=ALPHA)

    emp_cov = float(np.mean([y_ps[i, y_test.iloc[i], 0] for i in range(len(y_test))]))
    p0 = float(np.mean((y_ps[:, 0, 0] == 1) & (y_ps[:, 1, 0] == 0)))
    p1 = float(np.mean((y_ps[:, 0, 0] == 0) & (y_ps[:, 1, 0] == 1)))
    p01 = float(np.mean((y_ps[:, 0, 0] == 1) & (y_ps[:, 1, 0] == 1)))
    p_empty = float(np.mean(y_ps[:, :, 0].sum(axis=1) == 0))

    print(
        f"\n--- CONFORMAL PREDICTION (MAPIE) ON {dataset_label} ---\n"
        f"Target Coverage (1 - alpha): {1 - ALPHA:.1%} | Empirical Coverage: {emp_cov:.1%}\n"
        f"  - Low Risk Only {{0}}:    {p0 * 100:6.2f}% (Safe to discharge)\n"
        f"  - High Risk Only {{1}}:   {p1 * 100:6.2f}% (Definite intervention)\n"
        f"  - Uncertain {{0, 1}}:     {p01 * 100:6.2f}% (FLAG FOR DOCTOR CONSULTATION)\n"
        f"  - Empty Sets {{}}:        {p_empty * 100:6.2f}%"
    )
    return emp_cov, [p0 * 100, p1 * 100, p01 * 100]


cov_A, pcts_A = run_conformal_eval(
    pipes_A["Random_Forest"],
    X_val_A,
    y_val_A,
    X_test_A,
    y_test_A,
    "Dataset A (Framingham)",
)
cov_B, pcts_B = run_conformal_eval(
    pipes_B["Random_Forest"], X_val_B, y_val_B, X_test_B, y_test_B, "Dataset B (Cardio)"
)

for m in ["Logistic_Regression", "XGBoost"]:
    run_conformal_eval(
        pipes_A[m], X_val_A, y_val_A, X_test_A, y_test_A, f"Dataset A ({m})"
    )

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
cats = ["Low Risk\n{0}", "High Risk\n{1}", "Uncertain {0, 1}\n(Doctor Referral)"]
configs = [
    (axes[0], pcts_A, cov_A, "Dataset A: Framingham (Imbalanced 15.2%)"),
    (axes[1], pcts_B, cov_B, "Dataset B: Cardio (Balanced 49.5%)"),
]

for ax, pcts, cov, title in configs:
    bars = ax.bar(
        cats, pcts, color=["#2ca02c", "#d62728", "#ff7f0e"], alpha=0.85, width=0.5
    )
    ax.set(
        ylabel="Percentage of Test Population (%)",
        ylim=[0, 100],
        title=f"{title}\n(Empirical Coverage: {cov * 100:.1f}% >= 90%)",
    )
    for bar in bars:
        yval = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            yval + 2,
            f"{yval:.1f}%",
            ha="center",
            va="bottom",
            fontsize=11,
            fontweight="bold",
        )

plt.tight_layout()
save_fig(fig, "fig11_conformal_prediction_coverage.png")
plt.show()

## 5. Kết Luận Khoa Học, Thảo Luận & Khuyến Nghị Triển Khai Y Tế

---

### 1. Tổng Kết Kết Quả Nghiên Cứu Thực Nghiệm

1. **Hiệu Năng Mô Hình Nội Miền (In-Domain Assessment)**:
   - **Tập A (Framingham - Dữ liệu mất cân bằng $15.19\%$)**: Các mô hình tuyến tính và hạt nhân dẫn đầu về năng lực phân tách. **SVM RBF** (với `class_weight="balanced"`, $C=1.0, \gamma=0.01$) đạt Test ROC-AUC **0.747** [$95\%$ CI: $0.698 - 0.794$], PR-AUC **0.388**, Brier Score **0.114**. **Logistic Regression** ($C=0.1$) theo sát với ROC-AUC **0.746** [$0.698 - 0.792$] và Brier Score **0.113**. Các mô hình ensemble (Random Forest đạt **0.739**, XGBoost đạt **0.733**) có xu hướng chịu ảnh hưởng bởi kích thước mẫu nhỏ ($N=4.240$).
   - **Tập B (Cardio - Dữ liệu cân bằng $49.47\%$, $N=68.619$ sau làm sạch)**: Các mô hình phi tuyến ensemble thể hiện ưu thế vượt trội khi quy mô dữ liệu lớn. **XGBoost** đạt Test ROC-AUC **0.798** [$0.790 - 0.806$], PR-AUC **0.781**, Brier Score **0.180**. **Random Forest** đạt ROC-AUC **0.797** [$0.790 - 0.805$]. Đặc biệt, giải pháp xấp xỉ hạt nhân **Nyström + SGD** (`log_loss`) đạt ROC-AUC **0.795**, giải quyết triệt để rào cản tính toán $O(N^3)$ của Kernel SVM truyền thống mà vẫn bảo toàn độ chính xác.
   - **Ý Nghĩa Lâm Sàng Của Tối Ưu Hóa Ngưỡng ($p^*$)**: Việc dịch chuyển ngưỡng quyết định từ $0.50$ sang $p^*$ theo tiêu chí $\text{Recall} \ge 0.80$ giúp cắt giảm **4.6% đến 19.8%** tổng chi phí sai số y tế trên Tập A và **22.0% đến 34.8%** trên Tập B, bảo vệ bệnh nhân khỏi nguy cơ bị bỏ sót bệnh.
   - **Diễn Giải Dịch Tễ Học Qua Odds Ratio (Logistic Regression)**:
     - Trên Tập A: `age` ($\text{OR/1SD} = 1.7324$), `is_male` ($\text{OR} = 1.5327$), `sys_bp` ($\text{OR/1SD} = 1.4071$) và `smoker` ($\text{OR} = 1.3993$) là các yếu tố tương quan thuận mạnh nhất với nguy cơ biến cố 10 năm.
     - Trên Tập B: `sys_bp` ($\text{OR/1SD} = 2.6420$) và `chol_level` ($\text{OR/1SD} = 1.4012$) áp đảo hoàn toàn, phản ánh sự biểu hiện trực tiếp của tổn thương tim mạch hiện hữu.

2. **Hiện Tượng Trôi Dạt Dữ Liệu & Bài Học Về Chuyển Giao Xuyên Miền (Cross-Domain Transfer)**:
   - **Sự Suy Giảm Cực Đoan Của Mô Hình Zero-Shot**: Khi đưa mô hình XGBoost huấn luyện từ Tập A sang dự báo trên Tập B tại ngưỡng $p=0.50$, độ nhạy Recall sụp đổ xuống chỉ còn **0.93%** và Brier Score vọt lên **0.3460**. Ngược lại, khi chuyển từ $B \to A$, việc giữ nguyên ngưỡng khiến Dương tính giả bùng nổ, làm chi phí y tế gia tăng.
   - **Bằng Chứng Định Lượng Từ Adversarial Validation**: Mô hình phân loại miền đạt **ROC-AUC = 0.9841**, khẳng định hai tập dữ liệu bị trôi phân phối cực kỳ nghiêm trọng. Các đặc trưng đóng góp lớn nhất vào sự phân ly này là `chol_level` ($47.6\%$), `smoker` ($11.4\%$) và `dia_bp` ($10.1\%$).
   - **Khắc Phục Bằng Hiệu Chuẩn Xác Suất Hậu Nghiệm Hai Chiều**:
     - Hướng $A \to B$ (Khắc phục _Under-confidence_): Isotonic Calibration hạ Brier Score từ **0.3460 xuống 0.2138** (giảm **38.2%**); khi kết hợp với tái tối ưu ngưỡng đích ($p^* = 0.351$), Recall được phục hồi lên **85.95%** và chi phí y tế giảm **73.5%**.
     - Hướng $B \to A$ (Khắc phục _Over-confidence_): Platt Scaling (Sigmoid) hạ Brier Score từ **0.3310 xuống 0.1235** (giảm **62.7%**); kết hợp tái tối ưu ngưỡng đích ($p^* = 0.093$) khôi phục Recall lên **86.82%**.
   - **Giới Hạn Bản Chất Của Dịch Chuyển Khái Niệm (Concept Shift)**: Hiệu chuẩn hậu nghiệm giải quyết xuất sắc Base-rate Shift và độ lệch điểm cắt, nhưng không thể xóa bỏ sự khác biệt về quy luật $P(Y \mid X)$ giữa bài toán tiên lượng 10 năm và chẩn đoán tức thời. Điều này khẳng định tầm quan trọng của việc hiểu rõ định nghĩa lâm sàng trước khi triển khai AI.

3. **Khung Dự Đoán Tin Cậy & Minh Bạch Hóa Mô Hình (Trustworthy AI)**:
   - **Khám Phá Bản Chất Qua SHAP**: Phân tích SHAP giải thích thỏa đáng sự khác biệt cơ chế bệnh sinh: Tuổi tác tích lũy nguy cơ trong dài hạn (chi phối Tập A), trong khi Huyết áp và Cholesterol phản ánh bệnh lý cấp/mãn tính đang diễn ra (chi phối Tập B).
   - **Bài Học Cảnh Tỉnh Từ Ca Âm Tính Giả #672**: Biểu đồ SHAP Waterfall chỉ ra rằng mô hình bị đánh lừa bởi độ tuổi trẻ ($35$ tuổi, kéo SHAP về miền âm mạnh), dẫn đến việc đánh giá thấp nguy cơ dù bệnh nhân là nam giới hút thuốc có huyết áp bất thường. Đây là cơ sở để thiết lập các quy tắc can thiệp lâm sàng độc lập (Clinical Guardrails).
   - **Bảo Chứng Thống Kê Với Conformal Prediction**: Với mức ý nghĩa $\alpha = 0.10$, hệ thống đảm bảo độ bao phủ thực nghiệm vượt mức cam kết $90\%$ (**90.8%** trên Framingham và **90.1%** trên Cardio với Random Forest). Khả năng tự động cách ly các ca bất định $\{0, 1\}$ để bác sĩ hội chẩn mang lại giải pháp an toàn tuyệt đối cho quy trình tầm soát y tế.

---

### 2. Danh Mục Các Minh Chứng Thực Nghiệm Đã Xuất Bản (`figures/`)

- `fig1_eda_distributions.png`: Khảo sát phân phối sinh trắc và lâm sàng giữa hai quần thể.
- `fig2_correlation_matrix.png`: Cấu trúc ma trận tương quan nội miền của từng tập dữ liệu.
- `fig3_roc_pr_curves_framingham.png`: Đường cong ROC & PR của 7 mô hình trên Tập A.
- `fig4_roc_pr_curves_cardio.png`: Đường cong ROC & PR của 7 mô hình trên Tập B.
- `fig5_learning_curves.png`: Động thái hội tụ Bias-Variance qua 5-Fold Stratified CV.
- `fig6_in_domain_comparison.png`: Đối sánh hiệu năng tổng hợp (ROC-AUC & F1-Score) trên 2 tập.
- `fig7_adversarial_validation_drift.png`: Định lượng mức độ trôi dạt đặc trưng bằng Kiểm định Đối nghịch.
- `fig8_calibration_curves.png`: Đường cong độ tin cậy hiệu chuẩn xác suất hai chiều ($A \to B$ và $B \to A$).
- `fig9_shap_global_summary.png`: Đối sánh cơ chế quyết định toàn cục SHAP giữa Mỹ và Đông Âu.
- `fig10_shap_local_false_negatives.png`: Truy nguyên nguyên nhân bỏ sót bệnh qua SHAP Waterfall.
- `fig11_conformal_prediction_coverage.png`: Phân bổ tập dự đoán tin cậy theo phân phối Conformal Prediction.
